# 01 — Problématique, inventaire des données et fiches d'identification

**Projet NutriHFr** · Étapes 1 et 2 de la consigne

Ce notebook :
1. formule la question scientifique et les questions concrètes du projet ;
2. inventorie les sources disponibles et évalue leur volumétrie et leur qualité ;
3. **inspecte les colonnes des fichiers bruts** pour paramétrer `src/nutrihfr/config.py` ;
4. génère une fiche d'identification par jeu de données (`docs/fiches_identification.md`) ;
5. justifie le choix des jeux de données retenus.

## 1. Problème scientifique

> **Comment se caractérisent les consommations alimentaires et les apports nutritionnels des adultes vivant en France métropolitaine, comment varient-ils selon l'âge et le sexe, et quels aliments y contribuent le plus ?**

En langage clair : on cherche à décrire *ce que mangent* les Français adultes, *ce que cela leur apporte* en énergie et en nutriments, *qui* mange différemment (hommes/femmes, jeunes/âgés), et *d'où viennent* les nutriments (quels groupes d'aliments).

| Code | Question concrète | Exemple de réponse attendue |
|---|---|---|
| Q1 | Quelle est la distribution de l'apport énergétique journalier ? | moyenne pondérée et IC 95 %, histogramme |
| Q2 | Les apports diffèrent-ils entre hommes et femmes ? | écart de moyennes pondérées et IC |
| Q3 | Comment évoluent-ils avec l'âge ? | moyennes par classe d'âge, régression |
| Q4 | À énergie comparable, qui a l'alimentation la plus dense en nutriments ? | apports pour 1 000 kcal |
| Q5 | Quels aliments contribuent le plus aux protéines, au calcium, au fer, aux fibres, à la vitamine C ? | % de contribution par groupe |
| Q6 | Quelle est la variabilité entre individus d'un même groupe ? | P5–P95, violin plot, part intra/inter |

**Périmètre retenu** : adultes de 18 à 79 ans (les enfants ont des besoins et un mode de recueil différents ; on les exclut pour garder un projet cohérent).

In [ ]:
# --- Configuration commune à tous les notebooks ------------------------------
import sys
import warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=RuntimeWarning)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nutrihfr import config as C
from nutrihfr.viz import set_style, savefig, PALETTE, SEX_COLORS
set_style()
if C.MODE == "synthetic" and not C.raw_dir("inca3").exists():   # 1re utilisation : données de démo
    import runpy; runpy.run_path(str(ROOT / "scripts" / "make_synthetic_data.py"), run_name="__main__")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 50)
print("Mode des données :", C.MODE, "->", C.raw_dir("inca3"))

## 2. Sources disponibles

| Source | Producteur | Période | Contenu | Accès | Retenue ? |
|---|---|---|---|---|---|
| INCA 1 | AFSSA | 1998-1999 | consommations 7 jours | archive | **Non** : nomenclature et méthode incompatibles |
| INCA 2 | AFSSA/Anses | 2006-2007 | carnet 7 jours, apports (38 nutriments) | [data.gouv.fr](https://www.data.gouv.fr/datasets/donnees-de-consommations-et-habitudes-alimentaires-de-letude-inca-2-3) | **Oui, en complément** (bonus Q_Comp_Conso) |
| INCA 3 | Anses | 2014-2015 | 3 rappels de 24 h, consommations détaillées + composition + apports | [data.gouv.fr](https://www.data.gouv.fr/datasets/donnees-de-consommations-et-habitudes-alimentaires-de-letude-inca-3) | **Oui, source principale** |
| NutriNet-Santé | Inserm/INRAE… | depuis 2009 | cohorte en ligne | sur demande | Non (non ouvert) |
| Ciqual | Anses | 2020, 2025 | composition de ~3 000 aliments génériques | [ciqual.anses.fr](https://ciqual.anses.fr) | **Oui** : contrôles (Atwater) et mapping (bonus) |
| Open Food Facts | collaboratif | continu | produits de marque | ouvert | Non : produits de marque ≠ aliments génériques INCA |

### Pourquoi ne pas fusionner INCA 2 et INCA 3 ?

| Critère | INCA 2 | INCA 3 | Conséquence |
|---|---|---|---|
| Recueil | carnet sur 7 jours consécutifs | 3 rappels de 24 h non consécutifs | variabilité intra-individuelle différente |
| Population | 3-79 ans | 0-79 ans | restreindre aux 18-79 ans |
| Nomenclature | nomenclature INCA 2 | nomenclature INCA 3 (FoodEx2) | pas de correspondance 1-1 des aliments |
| Composition | version Ciqual de l'époque | version Ciqual plus récente | différences d'apports dues aux tables |
| Pondérations | propres à l'enquête | propres à l'enquête | estimer séparément, ne jamais empiler les poids |

L'Anses indique elle-même que ce changement méthodologique ne permet pas d'étudier précisément les évolutions entre INCA 2 et INCA 3. **Décision** : chaque enquête est analysée **séparément** ; INCA 2 sert uniquement à une comparaison prudente des apports au niveau agrégé (notebook 06).

## 3. Inventaire des fichiers : volumétrie et qualité globale

En mode `real`, dépose les CSV téléchargés dans `data/raw/inca3/`, `data/raw/inca2/` et `data/raw/ciqual/`, puis passe `MODE = "real"` dans `config.py`.

In [ ]:
from nutrihfr.loader import read_csv_robust, file_sha256, suggest_columns

def inventory(folder):
    rows = []
    for f in sorted(Path(folder).glob("*")):
        if f.suffix.lower() not in (".csv", ".xls", ".xlsx"):
            continue
        df = read_csv_robust(f)
        rows.append({"dossier": Path(folder).name, "fichier": f.name,
                     "taille_Mo": f.stat().st_size / 1e6, "lignes": len(df), "colonnes": df.shape[1],
                     "pct_cellules_vides": 100 * df.isna().mean().mean(),
                     "lignes_dupliquees": int(df.duplicated().sum()),
                     "sha256": file_sha256(f)[:12]})
    return rows

inv = pd.DataFrame(inventory(C.raw_dir("inca3")) + inventory(C.raw_dir("inca2"))
                   + inventory(C.raw_dir("ciqual")))
inv

## 4. Inspection des colonnes → paramétrage de `config.py`

**Étape clé avec les vraies données.** Les cellules suivantes affichent les colonnes de chaque fichier et cherchent celles qui ressemblent aux variables dont on a besoin. Corrige ensuite les noms marqués `# A VERIFIER` dans `config.py`.

In [ ]:
raw = {k: read_csv_robust(C.raw_dir("inca3") / f) for k, f in C.INCA3_FILES.items()}
for k, df in raw.items():
    print(f"\n=== {C.INCA3_FILES[k]}  ({df.shape[0]} lignes x {df.shape[1]} colonnes)")
    print(list(df.columns)[:60], "..." if df.shape[1] > 60 else "")

In [ ]:
keywords = ["NOIND", "sex", "age", "pond", "poids", "taille", "jour", "code", "qte",
            "gpe", "nrj", "kcal", "prot", "glu", "lip", "fib", "alc", "calc", "fer", "vit"]
for k, df in raw.items():
    hits = {kw: v for kw, v in suggest_columns(df, keywords).items() if v}
    print(f"\n--- {k}"); [print(f"  {kw:>6} -> {v[:8]}") for kw, v in hits.items()]

In [ ]:
# Vérification : chaque colonne attendue par config.py existe-t-elle dans le fichier ?
checks = []
pairs = [("indiv", "indiv"), ("conso", "conso"), ("conso", "conso_nutrients"),
         ("nomenclature", "nomenclature"), ("apports", "apports"), ("apports", "apports_nutrients")]
for file_key, map_key in pairs:
    for col_brut, col_canon in C.INCA3_COLS[map_key].items():
        checks.append({"fichier": C.INCA3_FILES[file_key], "colonne_brute": col_brut,
                       "nom_canonique": col_canon, "trouvee": col_brut in raw[file_key].columns})
checks = pd.DataFrame(checks)
print(f"{(~checks.trouvee).sum()} colonne(s) à corriger dans config.py")
checks[~checks.trouvee]

In [ ]:
# Codage du sexe et des classes d'âge (à reporter dans SEX_CODES / AGE_FROM_CLASS)
ind_raw = raw["indiv"]
for col in [c for c in ("sex_PS", "tage_PS") if c in ind_raw]:
    print(ind_raw[col].value_counts(dropna=False).sort_index(), "\n")

## 5. Fiches d'identification

Une fiche par jeu de données : métadonnées (renseignées à la main à partir des pages sources et des notices) + indicateurs calculés automatiquement. Le résultat est écrit dans `docs/fiches_identification.md`.

In [ ]:
META = {
    "INCA 3": dict(
        producteur="Anses", url="https://www.data.gouv.fr/datasets/donnees-de-consommations-et-habitudes-alimentaires-de-letude-inca-3",
        periode="2014-2015", population="0-79 ans, France métropolitaine (≈ 5 800 individus, à confirmer dans la notice)",
        methode="3 rappels de 24 h non consécutifs (adultes) ; enregistrements pour les enfants",
        granularite="ligne = un aliment consommé, un individu, un jour", cle="NOIND (+ nojour)",
        licence="non précisée sur data.gouv.fr : respecter les règles de citation de la notice utilisateurs",
        version="mise à jour du 28/01/2021", rgpd="données pseudonymisées (identifiant NOIND, pas de nom)",
        limites="sous-déclaration possible, 3 jours seulement, composition figée à la version Ciqual utilisée par l'Anses",
        dossier=C.raw_dir("inca3")),
    "INCA 2": dict(
        producteur="AFSSA / Anses", url="https://www.data.gouv.fr/datasets/donnees-de-consommations-et-habitudes-alimentaires-de-letude-inca-2-3",
        periode="2006-2007", population="3-79 ans, France métropolitaine (≈ 2 600 adultes)",
        methode="carnet alimentaire sur 7 jours consécutifs", granularite="individu (apports) / aliment-jour (consommations)",
        cle="identifiant individu (nomen)", licence="Licence Ouverte (Etalab)", version="mise en ligne 24/09/2014",
        rgpd="données pseudonymisées", limites="méthode et nomenclature différentes d'INCA 3",
        dossier=C.raw_dir("inca2")),
    "Ciqual": dict(
        producteur="Anses", url="https://ciqual.anses.fr", periode=C.CIQUAL_VERSION,
        population="≈ 3 000 aliments génériques", methode="compilation d'analyses et de données bibliographiques",
        granularite="ligne = un aliment, valeurs pour 100 g", cle="alim_code", licence="Licence Ouverte (Etalab)",
        version=C.CIQUAL_VERSION + " (une version 2025 existe : documenter le choix)", rgpd="sans objet (pas de données personnelles)",
        limites="valeurs manquantes ou texte ('traces', '< x'), versions successives non comparables à 100 %",
        dossier=C.raw_dir("ciqual")),
}

def fiche(nom, m):
    lines = [f"## {nom}", "", "| Rubrique | Contenu |", "|---|---|"]
    labels = {"producteur": "Producteur", "url": "URL", "periode": "Période", "population": "Population",
              "methode": "Méthode de recueil", "granularite": "Granularité", "cle": "Clé(s)",
              "licence": "Licence", "version": "Version", "rgpd": "Confidentialité (RGPD)",
              "limites": "Limites connues"}
    for k, lab in labels.items():
        lines.append(f"| {lab} | {m[k]} |")
    sub = inv[inv["dossier"] == Path(m["dossier"]).name]
    lines += ["", "**Volumétrie et qualité (calculées)**", "",
              "| Fichier | Lignes | Colonnes | Taille (Mo) | % cellules vides | Lignes dupliquées | SHA-256 |",
              "|---|---|---|---|---|---|---|"]
    for r in sub.itertuples():
        lines.append(f"| {r.fichier} | {r.lignes:,} | {r.colonnes} | {r.taille_Mo:.2f} | "
                     f"{r.pct_cellules_vides:.1f} | {r.lignes_dupliquees} | `{r.sha256}` |")
    return "\n".join(lines) + "\n"

text = "# Fiches d'identification des jeux de données\n\n"
text += f"_Générées par le notebook 01 (mode `{C.MODE}`)._\n\n" + "\n".join(fiche(n, m) for n, m in META.items())
(ROOT / "docs" / "fiches_identification.md").write_text(text, encoding="utf-8")
print(text[:1500])

## 6. Choix des données (conclusion)

- **Source principale : INCA 3**, adultes 18-79 ans. C'est la seule enquête qui fournit à la fois les consommations détaillées, la composition des aliments et les apports calculés par l'Anses, ce qui permet de **valider notre pipeline** en recalculant les apports.
- **Composition** : celle fournie dans `conso-compo-alim` (cohérente avec la version de table utilisée par l'Anses pour INCA 3). Ciqual, en version récente, sert de référence externe pour les contrôles (cohérence Atwater) et pour le bonus de mapping automatique.
- **INCA 2** : utilisé seul, pour une comparaison agrégée et prudente avec INCA 3 (notebook 06), jamais fusionné.

➡️ Suite : `02_data_engineering.ipynb`